# Online Retail Sales Analysis
**Dataset:** Online Retail UCI | **Tools:** Python, SQL, Power BI

End-to-end data analysis covering:
- Data cleaning
- Exploratory data analysis
- SQL business queries
- RFM customer segmentation
- Export for Power BI

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import sqlite3
import datetime
import os

os.makedirs('visuals', exist_ok=True)
os.makedirs('data', exist_ok=True)

print('Libraries loaded successfully')

## Step 1 — Load Data

In [ ]:
df = pd.read_csv('data/online_retail.csv', encoding='unicode_escape')
print(f'Shape: {df.shape}')
df.head()

In [ ]:
print('Data types:')
print(df.dtypes)
print('\nMissing values:')
print(df.isnull().sum())
print(f'\nDuplicates: {df.duplicated().sum()}')

## Step 2 — Clean Data

In [ ]:
# Drop missing CustomerID
df = df.dropna(subset=['CustomerID'])

# Remove cancelled orders
df = df[~df['InvoiceNo'].astype(str).str.startswith('C')]

# Remove invalid quantities and prices
df = df[df['Quantity'] > 0]
df = df[df['UnitPrice'] > 0]

# Remove duplicates
df = df.drop_duplicates()

# Fix types
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])
df['CustomerID'] = df['CustomerID'].astype(int)

# Feature engineering
df['Revenue'] = df['Quantity'] * df['UnitPrice']
df['Year'] = df['InvoiceDate'].dt.year
df['Month'] = df['InvoiceDate'].dt.month
df['YearMonth'] = df['InvoiceDate'].dt.to_period('M')
df['DayOfWeek'] = df['InvoiceDate'].dt.day_name()

print(f'Cleaned shape: {df.shape}')
df.head()

## Step 3 — Exploratory Data Analysis

In [ ]:
# Monthly Revenue Trend
monthly_revenue = df.groupby('YearMonth')['Revenue'].sum().reset_index()
monthly_revenue['YearMonth'] = monthly_revenue['YearMonth'].astype(str)

plt.figure(figsize=(12, 5))
plt.plot(monthly_revenue['YearMonth'], monthly_revenue['Revenue'], marker='o', color='steelblue')
plt.title('Monthly Revenue Trend')
plt.xlabel('Month')
plt.ylabel('Revenue (GBP)')
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig('visuals/monthly_revenue.png')
plt.show()

In [ ]:
# Top 10 Products by Revenue
top_products = df.groupby('Description')['Revenue'].sum().sort_values(ascending=False).head(10)

plt.figure(figsize=(10, 6))
top_products.plot(kind='barh', color='coral')
plt.title('Top 10 Products by Revenue')
plt.xlabel('Revenue (GBP)')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.savefig('visuals/top_products.png')
plt.show()

In [ ]:
# Top 10 Countries by Revenue
top_countries = df.groupby('Country')['Revenue'].sum().sort_values(ascending=False).head(10)

plt.figure(figsize=(10, 6))
top_countries.plot(kind='bar', color='mediumseagreen')
plt.title('Top 10 Countries by Revenue')
plt.xlabel('Country')
plt.ylabel('Revenue (GBP)')
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig('visuals/top_countries.png')
plt.show()

In [ ]:
# Orders by Day of Week
day_order = ['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday']
orders_by_day = df.groupby('DayOfWeek')['InvoiceNo'].nunique().reindex(day_order)

plt.figure(figsize=(8, 5))
orders_by_day.plot(kind='bar', color='mediumpurple')
plt.title('Number of Orders by Day of Week')
plt.xlabel('Day')
plt.ylabel('Number of Orders')
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig('visuals/orders_by_day.png')
plt.show()

In [ ]:
# Distribution of Order Values
order_value = df.groupby('InvoiceNo')['Revenue'].sum()

plt.figure(figsize=(10, 5))
order_value[order_value < 500].hist(bins=50, color='steelblue', edgecolor='white')
plt.title('Distribution of Order Values (under £500)')
plt.xlabel('Order Value (GBP)')
plt.ylabel('Frequency')
plt.tight_layout()
plt.savefig('visuals/order_distribution.png')
plt.show()

print(f'Median order value: £{order_value.median():.2f}')
print(f'Mean order value:   £{order_value.mean():.2f}')

In [ ]:
# Quantity vs Unit Price
plt.figure(figsize=(8, 5))
plt.scatter(df['Quantity'], df['UnitPrice'], alpha=0.3, color='tomato', s=5)
plt.title('Quantity vs Unit Price')
plt.xlabel('Quantity')
plt.ylabel('Unit Price (GBP)')
plt.xlim(0, 100)
plt.ylim(0, 50)
plt.tight_layout()
plt.savefig('visuals/quantity_vs_price.png')
plt.show()

print(f"Correlation between Quantity and Unit Price: {df['Quantity'].corr(df['UnitPrice']):.4f}")

## Step 4 — SQL Analysis

In [ ]:
# Load into SQLite
df_sql = df.copy()
df_sql['YearMonth'] = df_sql['YearMonth'].astype(str)
df_sql['InvoiceDate'] = df_sql['InvoiceDate'].astype(str)

conn = sqlite3.connect('data/retail.db')
df_sql.to_sql('retail', conn, if_exists='replace', index=False)
print('Data loaded into SQLite')

In [ ]:
# Overall Summary
pd.read_sql_query("""
    SELECT
        COUNT(DISTINCT InvoiceNo)  AS total_orders,
        COUNT(DISTINCT CustomerID) AS total_customers,
        ROUND(SUM(Revenue), 2)     AS total_revenue,
        ROUND(AVG(Revenue), 2)     AS avg_order_value
    FROM retail
""", conn)

In [ ]:
# Top 10 Products
pd.read_sql_query("""
    SELECT Description, SUM(Quantity) AS units_sold, ROUND(SUM(Revenue),2) AS revenue
    FROM retail
    GROUP BY Description
    ORDER BY revenue DESC
    LIMIT 10
""", conn)

In [ ]:
# Monthly Revenue Growth Rate (Window Function)
pd.read_sql_query("""
    WITH monthly AS (
        SELECT YearMonth, ROUND(SUM(Revenue), 2) AS revenue
        FROM retail
        GROUP BY YearMonth
    )
    SELECT
        YearMonth,
        revenue,
        LAG(revenue) OVER (ORDER BY YearMonth) AS prev_month,
        ROUND(
            (revenue - LAG(revenue) OVER (ORDER BY YearMonth)) * 100.0
            / LAG(revenue) OVER (ORDER BY YearMonth), 2
        ) AS growth_pct
    FROM monthly
""", conn)

In [ ]:
# Top 10 Customers
pd.read_sql_query("""
    SELECT CustomerID, COUNT(DISTINCT InvoiceNo) AS orders, ROUND(SUM(Revenue),2) AS total_spent
    FROM retail
    GROUP BY CustomerID
    ORDER BY total_spent DESC
    LIMIT 10
""", conn)

In [ ]:
conn.close()

## Step 5 — RFM Customer Segmentation

In [ ]:
reference_date = df['InvoiceDate'].max() + datetime.timedelta(days=1)

rfm = df.groupby('CustomerID').agg(
    Recency=('InvoiceDate', lambda x: (reference_date - x.max()).days),
    Frequency=('InvoiceNo', 'nunique'),
    Monetary=('Revenue', 'sum')
).reset_index()

rfm['Monetary'] = rfm['Monetary'].round(2)

rfm['R_Score'] = pd.qcut(rfm['Recency'], 4, labels=[4, 3, 2, 1])
rfm['F_Score'] = pd.qcut(rfm['Frequency'].rank(method='first'), 4, labels=[1, 2, 3, 4])
rfm['M_Score'] = pd.qcut(rfm['Monetary'], 4, labels=[1, 2, 3, 4])

def segment(row):
    score = int(row['R_Score']) + int(row['F_Score']) + int(row['M_Score'])
    if score >= 10:
        return 'Champions'
    elif score >= 7:
        return 'Loyal Customers'
    elif score >= 5:
        return 'At Risk'
    else:
        return 'Lost'

rfm['Segment'] = rfm.apply(segment, axis=1)

print(rfm['Segment'].value_counts())
rfm.head(10)

In [ ]:
# Segment Bar Chart
segment_counts = rfm['Segment'].value_counts()

plt.figure(figsize=(8, 5))
segment_counts.plot(kind='bar', color=['#2ecc71', '#3498db', '#e67e22', '#e74c3c'])
plt.title('Customer Segments (RFM)')
plt.xlabel('Segment')
plt.ylabel('Number of Customers')
plt.xticks(rotation=30)
plt.tight_layout()
plt.savefig('visuals/rfm_segments.png')
plt.show()

In [ ]:
# Segment Summary
rfm.groupby('Segment').agg(
    Customers=('CustomerID', 'count'),
    Avg_Recency=('Recency', 'mean'),
    Avg_Frequency=('Frequency', 'mean'),
    Avg_Monetary=('Monetary', 'mean')
).round(2)

## Step 6 — Export for Power BI

In [ ]:
df_export = df.copy()
df_export['YearMonth'] = df_export['YearMonth'].astype(str)
df_export['InvoiceDate'] = df_export['InvoiceDate'].astype(str)
df_export.to_csv('data/cleaned_retail.csv', index=False)

rfm.to_csv('data/rfm_segments.csv', index=False)

monthly_summary = df.groupby('YearMonth').agg(
    Revenue=('Revenue', 'sum'),
    Orders=('InvoiceNo', 'nunique'),
    Customers=('CustomerID', 'nunique')
).reset_index()
monthly_summary['YearMonth'] = monthly_summary['YearMonth'].astype(str)
monthly_summary.to_csv('data/monthly_summary.csv', index=False)

print('All files exported to data/ folder')